<a href="https://colab.research.google.com/github/MonikaBarget/DigitalHistory/blob/master/FindZenodoDOIs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Code to match conference papers titles in an EXCEL table with Zenodo DOIs and URLs**

This code automatically matches conference paper titles from an EXCEL column to DOIs and URLs on Zenodo. In the example below, we want to match submission titles for the DH Benelux 2026 conference to presentation abstracts that the participants have uploaded to the conference's Zenodo community.

For this operation, we can simply work with the ```requests``` package in Python and do not need browser automation. Tools like ```selenium``` are only required when buttons need to be clicked or we need to enter user data. Our workflow only collects structured metadata via Zenodo’s REST endpoint, which returns the information in JSON format.

Registering for a Zenodo developer key is optional and only necessary if you need to expand the rate limites or want to write information (record updates) to Zenodo.

In [ ]:
# Package installations (silent mode)

!pip -q install pandas openpyxl requests thefuzz[speedup]

import pandas as pd
import requests
import time
from thefuzz import fuzz
from google.colab import files

print("Installations complete!")

In [ ]:
# Upload your EXCEL file
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_excel(file_name)
print(f"\nRead {len(df)} rows from spreadsheet.")

# Define your title column
TITLE_COLUMN = "Presentation Title" # Change this if needed!

if TITLE_COLUMN not in df.columns:
    raise ValueError(f"Column '{TITLE_COLUMN}' not found.")

# Define community and fetch available records
def fetch_zenodo_records(community="dhbenelux2026", max_pages=10):
    records = []

    for page in range(1, max_pages + 1):
        print(f"\nFetching page {page}")

        url = "https://zenodo.org/api/records"

        params = {
            "q": f"communities:{community}",
            "page": page,
            "size": 25
        }

        r = requests.get(url, params=params)

        if r.status_code != 200:
            print("API error:", r.status_code)
            break

        hits = r.json().get("hits", {}).get("hits", [])

        if not hits:
            break

        for item in hits:
            metadata = item.get("metadata", {})

            title = metadata.get("title", "").strip()
            doi = metadata.get("doi", "").strip()

            records.append({
                "title": title,
                "doi": doi
            })

        time.sleep(0.3)

    return records

print("Fetching Zenodo records...")
zenodo_records = fetch_zenodo_records()

print(f"Found {len(zenodo_records)} records.")

# Find DOIs in format 10.5281/zenodo.19235931 and
# deduct record URLs in format https://zenodo.org/records/19235931

def doi_to_zenodo_url(doi):
    if not doi or doi == "No DOI yet.":
        return ""

    doi = str(doi).strip()

    if "zenodo." not in doi:
        return ""

    record_id = doi.split("zenodo.")[-1]
    return f"https://zenodo.org/records/{record_id}"

# Match titles using a fuzzy algorithm to capture deviations
def find_best_match(title, records, threshold=70):
    best_doi = None
    best_title = ""
    best_score = 0

    for record in records:
        if not record["title"]:
            continue

        score = fuzz.token_sort_ratio(
            str(title).lower(),
            record["title"].lower()
        )

        if score > best_score and score >= threshold:
            best_score = score
            best_doi = record["doi"]
            best_title = record["title"]

    if best_doi:
        return pd.Series([
            best_doi,
            doi_to_zenodo_url(best_doi),
            best_title,
            best_score
        ])

    return pd.Series([
        "No DOI yet.",
        "",
        "",
        0
    ])

# Apply matching and add results to dataframe

print("Matching titles...")

df[[
    "DOI",
    "Zenodo URL",
    "Matched Zenodo Title",
    "Match Score" # helps you trace considerable deviations!
]] = df[TITLE_COLUMN].apply(
    lambda x: find_best_match(x, zenodo_records)
)

# Save results to a new file
output_file = "updated_presentations_with_doi.xlsx"
df.to_excel(output_file, index=False)

files.download(output_file)

print("Done.")
print("Saved:", output_file)

You are welcome to reuse this code for your own conferences or other academic / educational purposes.